In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, MultiLabelBinarizer

# Load your data
df = pd.read_csv('connections_data.csv')
print("Initial Data Shape:", df.shape)
df.head()

Initial Data Shape: (100, 7)


,user_id,commodity,role,city,state,min_quantity_mt,max_quantity_mt
0,1,rice,trader,Nagpur,Maharashtra,50,200
1,2,sugar,broker,Nagpur,Maharashtra,20,100
2,3,cotton,exporter,Nagpur,Maharashtra,100,500
3,4,rice;sugar,trader,Nagpur,Maharashtra,30,150
4,5,cotton;sugar,broker,Nagpur,Maharashtra,40,180


In [6]:
CONFIG = {
    # Quantity Bounds (Metric Tons)
    'QTY_MIN': 0,
    'QTY_MAX': 10000, 
    
    # Variable Metrics Max Caps (from PDF logic)
    'FOLLOWERS_MAX': 100000,
    'ENGAGEMENT_MAX': 5000,
    'SCREENTIME_MAX': 10000, # Example from PDF [cite: 3]
    
    # Decay rate for 'Recent' metric
    'LAMBDA_DECAY': 0.05, # The "dial" mentioned in source 3 [cite: 3]
    
    # Role Affinity Weights (for the search/layer step later)
    'ROLE_AFFINITY': {
        'trader': {'broker': 0.5, 'exporter': 0.3, 'trader': 0.2},
        'broker': {'broker': 0.33, 'exporter': 0.33, 'trader': 0.33},
        'exporter': {'broker': 0.5, 'exporter': 0.2, 'trader': 0.3}
    }
}

In [7]:
# 1. Quantity Formula: f(x) = (x - min) / (max - min)
def global_min_max_scale(val, min_val, max_val):
    return (val - min_val) / (max_val - min_val)

# 2. Variable Metric Formula: V = ln(1 + x) / ln(1 + max)
def global_log_scale(val, max_val):
    return np.log(1 + val) / np.log(1 + max_val)

# Applying Quantity Normalization
df['min_qty_norm'] = df['min_quantity_mt'].apply(
    lambda x: global_min_max_scale(x, CONFIG['QTY_MIN'], CONFIG['QTY_MAX'])
)
df['max_qty_norm'] = df['max_quantity_mt'].apply(
    lambda x: global_min_max_scale(x, CONFIG['QTY_MIN'], CONFIG['QTY_MAX'])
)


In [ ]:
# One-Hot and Multi-Hot setups
role_enc = OneHotEncoder(sparse_output=False)
city_enc = OneHotEncoder(sparse_output=False)
state_enc = OneHotEncoder(sparse_output=False)
mlb = MultiLabelBinarizer()

# Generating vectors
role_vecs = role_enc.fit_transform(df[['role']])
city_vecs = city_enc.fit_transform(df[['city']])
state_vecs = state_enc.fit_transform(df[['state']])
commodity_vecs = mlb.fit_transform(df['commodity'].apply(lambda x: [x.strip()]))

# Stack everything: Commodity + Role + Quantities + Followers + Engagement + Location
master_vectors = np.hstack([
    commodity_vecs, 
    role_vecs, 
    df[['min_qty_norm', 'max_qty_norm']].values,
    city_vecs,
    state_vecs
])



df['final_vector'] = master_vectors.tolist()
print(f"Final Vector Dimension: {master_vectors.shape[1]}")

Final Vector Dimension: 136


In [10]:
# Updated assembly: Excluding followers and engagement as requested
master_vectors_simple = np.hstack([
    commodity_vecs, 
    role_vecs, 
    df[['min_qty_norm', 'max_qty_norm']].values,
    city_vecs,
    state_vecs
])

# Create a clean display
for i, user_vec in enumerate(master_vectors_simple):
    print(f"User {i+1} Vector ({df.iloc[i]['role']} - {df.iloc[i]['commodity']}):")
    # Formatting to 3 decimal places for readability
    formatted_vec = [round(x, 3) for x in user_vec]
    print(formatted_vec)
    print("-" * 30)

User 1 Vector (trader - rice):
[np.float64(0.0), np.float64(0.0), np.float64(1.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(1.0), np.float64(0.005), np.float64(0.02), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.

In [11]:
import json

# Create a 'Vector DB ready' format: List of dictionaries
vector_records = []
for index, row in df.iterrows():
    record = {
        "user_id": int(row['user_id']),
        "vector": row['final_vector'],
        "metadata": {
            "commodity": row['commodity'],
            "role": row['role'],
            "city": row['city'],
            "state": row['state']
        }
    }
    vector_records.append(record)

# Save to file
with open('user_vectors.json', 'w') as f:
    json.dump(vector_records, f, indent=4)

print("Saved user_vectors.json successfully!")

Saved user_vectors.json successfully!


In [12]:
# Saving the dataframe with user_id and final_vector
df[['user_id', 'final_vector']].to_csv('user_vectors.csv', index=False)

print("Saved user_vectors.csv successfully!")

Saved user_vectors.csv successfully!


In [13]:
# Convert the column of lists into a pure 2D NumPy array
vector_matrix = np.array(df['final_vector'].tolist())

# Save as a .npy file
np.save('user_vectors_matrix.npy', vector_matrix)

print("Saved user_vectors_matrix.npy successfully!")

Saved user_vectors_matrix.npy successfully!


In [14]:
from sklearn.metrics.pairwise import cosine_similarity

# Convert your list of vectors into a matrix
all_vectors = np.array(df['final_vector'].tolist())

# Calculate similarity of User 1 (index 0) against everyone
user_1_sim = cosine_similarity([all_vectors[0]], all_vectors)[0]

# Add to dataframe to view
df['similarity_with_user_1'] = user_1_sim
print(df[['user_id', 'role', 'commodity', 'similarity_with_user_1']])

    user_id      role     commodity  similarity_with_user_1
0         1    trader          rice                1.000000
1         2    broker         sugar                0.500019
2         3  exporter        cotton                0.500073
3         4    trader    rice;sugar                0.750017
4         5    broker  cotton;sugar                0.500047
..      ...       ...           ...                     ...
95       96  exporter          rice                0.500067
96       97    trader   rice;cotton                0.500070
97       98    broker         sugar                0.250036
98       99  exporter          rice                0.500076
99      100    trader        cotton                0.500078

[100 rows x 4 columns]


In [15]:
# From your PDF:
# Trader prefers: Broker (0.5), Exporter (0.3), Trader (0.2)
# Broker prefers: Equal (0.33 each)
# Exporter prefers: Broker (0.5), Exporter (0.2), Trader (0.3)

AFFINITY_WEIGHTS = {
    'trader':   {'broker': 0.50, 'exporter': 0.30, 'trader': 0.20},
    'broker':   {'broker': 0.33, 'exporter': 0.33, 'trader': 0.33},
    'exporter': {'broker': 0.50, 'exporter': 0.20, 'trader': 0.30}
}

In [22]:
def calculate_commodity_score(user_a_list, user_b_list):
    set_a = set(user_a_list)
    set_b = set(user_b_list)
    
    common = set_a.intersection(set_b)
    all_unique = set_a.union(set_b)
    
    if not all_unique:
        return 0
    return len(common) / len(all_unique)

# 1. Identify User 1 (The Searcher)
user_1_idx = 0
user_1_row = df.iloc[user_1_idx]
user_1_commodities = user_1_row['commodity']
user_1_role = user_1_row['role']

# 2. Calculate scores for everyone else
results = []
for idx, row in df.iterrows():
    # Calculate Commodity Match (V)
    comm_score = calculate_commodity_score(user_1_commodities, row['commodity'])
    
    # Get Raw Vector Similarity (Location + Quantity + Role match)
    raw_sim = cosine_similarity([all_vectors[user_1_idx]], [all_vectors[idx]])[0][0]
    
    # Get Role Affinity Weight
    role_weight = AFFINITY_WEIGHTS[user_1_role].get(row['role'], 0.1)
    
    # Final Weighted Score for this step
    # We prioritize the Commodity Match as a multiplier to the affinity
    final_score = raw_sim * comm_score * role_weight
    
    results.append({
        'user_id': row['user_id'],
        'role': row['role'],
        'commodity': row['commodity'],
        'comm_match_val': round(comm_score, 3),
        'role_affinity': role_weight,
        'final_score': round(final_score, 4)
    })

# Display Results
df_final = pd.DataFrame(results).sort_values(by='final_score', ascending=False)
print(f"Results for User 1 ({user_1_role} dealing in {user_1_commodities}):")
print(df_final)

Results for User 1 (trader dealing in rice):
    user_id      role     commodity  comm_match_val  role_affinity  \
8         9  exporter          rice           1.000            0.3   
5         6  exporter          rice           1.000            0.3   
0         1    trader          rice           1.000            0.2   
10       11    trader          rice           1.000            0.2   
95       96  exporter          rice           1.000            0.3   
..      ...       ...           ...             ...            ...   
57       58    broker         sugar           0.125            0.5   
52       53  exporter        cotton           0.143            0.3   
44       45    broker  cotton;sugar           0.167            0.5   
47       48    broker         sugar           0.125            0.5   
51       52    broker         sugar           0.125            0.5   

    final_score  
8         0.225  
5         0.225  
0         0.200  
10        0.150  
95        0.150  
..    